In [1]:
import redis
import time
import json

In [2]:
# 连接 Redis
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

In [8]:
## 基于List的任务队列

# 生产者推送任务
def push_task(task):
    r.rpush("task_queue__", task)
    print(f"任务 {task} 已加入队列")

# 测试
push_task("task_1")
push_task("task_2")

# 消费者处理任务
def process_tasks():
    while True:
        task = r.blpop("task_queue__", timeout=10)  # 阻塞式等待任务
        print(type(task))
        print(task)
        if task:
            _, task_value = task
            print(f"_是:{_}")
            print(f"处理任务: {task_value}")
        else:
            print("无任务，等待中...")

# 启动消费者
process_tasks()



任务 task_1 已加入队列
任务 task_2 已加入队列


In [6]:
## 基于Pub/Sub（发布订阅）

def publish_message(channel, message):
    r.publish(channel, message)
    print(f"消息发送到 {channel}: {message}")

# 测试
publish_message("news", "Breaking news: Redis is amazing!")
publish_message("class", "Breaking class: Redis is amazing!")




消息发送到 news: Breaking news: Redis is amazing!
消息发送到 class: Breaking class: Redis is amazing!


In [19]:
pubsub = r.pubsub()
pubsub.subscribe("news")  # 订阅 news 频道

print("等待消息中...")
for message in pubsub.listen():
    if message['type'] == 'message':
        print(f"收到消息: {message['data']}")

等待消息中...


KeyboardInterrupt: 

In [13]:
GROUP_NAME = "workers"
CONSUMER_NAME = "worker_2"
STREAM_NAME = "task_stream"

while True:
    time.sleep(1)
    tasks = r.xreadgroup(GROUP_NAME, CONSUMER_NAME, {STREAM_NAME: ">"}, count=1, block=3000)
    print("任务数：",len(tasks))
    for stream, messages in tasks:
        for message_id, message in messages:
            print(f"消费者2 处理任务: {message['task']}")
            r.xack(STREAM_NAME, GROUP_NAME, message_id)  # 确认任务已处理

任务数： 1
消费者2 处理任务: stream_task_5
任务数： 1
消费者2 处理任务: stream_task_8
任务数： 0
任务数： 0
任务数： 0


KeyboardInterrupt: 